In [8]:
%%writefile requirements.txt
kagglehub
pandas
numpy
scikit-learn
matplotlib
joblib
fastapi
uvicorn
mlflow
dvc
pyyaml
sentence-transformers
rank-bm25
faiss-cpu
optuna
pandera
tqdm


Writing requirements.txt


In [9]:
%pip install -r requirements.txt


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.6/40.6 kB 2.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 91.7 MB/s eta 0:00:00:00:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.0/3.0 MB 98.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 72.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 469.8/469.8 kB 31.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 66.1 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 419.5/419.5 kB 27.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 303.6/303.6 kB 22.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 147.8/147.8 kB 12.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.3/79.3 kB 6.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 451.2/451.2 kB 19.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.8/41.8 kB 3.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━

In [12]:
import os
os.chdir('/Users/neecat/Desktop/Projects/movie-recommender')

FileNotFoundError: [Errno 2] No such file or directory: '/Users/neecat/Desktop/Projects/movie-recommender'

In [11]:
from __future__ import annotations

import inspect
import json
import os
import sys
import time
from pathlib import Path


def _find_project_root() -> Path:
    """Notebooks have no __file__; discover repo root from cwd (repo root or scripts/)."""
    here = Path.cwd().resolve()
    for p in [here, *here.parents]:
        if (p / "src" / "models").is_dir():
            return p
    raise RuntimeError(
        "Cannot find repo root (expected a folder with src/models). "
        "Run: import os; os.chdir('/path/to/movie-recommender') then re-run this cell."
    )


PROJECT_ROOT = _find_project_root()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import joblib
import mlflow
import pandas as pd

from src.models.hybrid import HybridRecommender
from src.models.reranker import train_reranker


def _func_defaults(func) -> dict:
    sig = inspect.signature(func)
    return {
        name: p.default
        for name, p in sig.parameters.items()
        if p.default is not inspect.Parameter.empty
    }


def main() -> None:
    project_root = PROJECT_ROOT
    processed_path = project_root / "data" / "processed" / "movies_processed.csv"
    models_dir = project_root / "models"
    models_dir.mkdir(parents=True, exist_ok=True)

    if not processed_path.exists():
        raise FileNotFoundError(f"Processed dataset not found: {processed_path}")

    print("Loading processed data...")
    df = pd.read_csv(processed_path)
    print(f"Rows loaded: {len(df)}")
    mlflow.set_tracking_uri(
        os.getenv("MLFLOW_TRACKING_URI")
    )
    mlflow.set_experiment("movie_recommender")

    embedding_cache = models_dir / "embeddings_all-mpnet-base-v2.npy"
    model = HybridRecommender(
        content_weight=0.20,
        embedding_weight=0.30,
        popularity_weight=0.10,
        bm25_weight=0.05,
        genre_weight=0.35,
        min_votes=500,
        use_embeddings=True,
        use_bm25=False,
        use_faiss=True,
        embedding_model="all-mpnet-base-v2",
        embedding_cache_path=str(embedding_cache),
    )
    with mlflow.start_run(run_name="hybrid_train"):
        mlflow.log_params(
            {
                "content_weight": model.content_weight,
                "embedding_weight": model.embedding_weight,
                "popularity_weight": model.popularity_weight,
                "bm25_weight": model.bm25_weight,
                "genre_weight": model.genre_weight,
                "min_votes": model.min_votes,
            }
        )
        print("Fitting hybrid model (this can take a few minutes)...")
        fit_start = time.time()
        model.fit(df)
        fit_duration = time.time() - fit_start
        print(f"Model fit completed in {fit_duration:.1f}s")

        print("Training reranker...")
        rr_defaults = _func_defaults(train_reranker)
        rr_start = time.time()
        reranker = train_reranker(df, model)
        reranker_duration = time.time() - rr_start
        model.set_reranker(reranker)
        print("Reranker trained.")

        reports_dir = project_root / "reports" / "results"
        reports_dir.mkdir(parents=True, exist_ok=True)
        training_metrics = {
            "fit_duration_sec": round(fit_duration, 3),
            "reranker_fit_duration_sec": round(reranker_duration, 3),
            "training_rows": len(df),
            "content_weight": model.content_weight,
            "embedding_weight": model.embedding_weight,
            "popularity_weight": model.popularity_weight,
            "bm25_weight": model.bm25_weight,
            "genre_weight": model.genre_weight,
            "min_votes": model.min_votes,
            "embedding_model": model.embedding_model,
            "use_embeddings": model.use_embeddings,
            "use_bm25": model.use_bm25,
            "use_faiss": model.use_faiss,
            "faiss_top_k": model.faiss_top_k,
            "reranker_sample_size": rr_defaults.get("sample_size"),
            "reranker_top_k": rr_defaults.get("top_k"),
        }
        (reports_dir / "training_metrics.json").write_text(
            json.dumps(training_metrics, indent=2)
        )
        mlflow.log_metrics(
            {
                "train_fit_duration_sec": fit_duration,
                "train_reranker_duration_sec": reranker_duration,
                "train_rows": len(df),
            }
        )

        artifacts = model.export_artifacts()
        artifacts["training_rows"] = len(df)
        artifacts["processed_path"] = str(processed_path)
        artifact_path = models_dir / "hybrid_artifacts.joblib"
        joblib.dump(artifacts, artifact_path)
        mlflow.log_artifact(str(artifact_path))
        print(f"Saved artifacts to: {artifact_path}")


if __name__ == "__main__":
    main()


RuntimeError: Cannot find repo root (expected a folder with src/models). Run: import os; os.chdir('/path/to/movie-recommender') then re-run this cell.